In [12]:
from pathlib import Path

import geopandas as gpd


RAIZ_PROYECTO = Path.cwd()
RUTA_BASE = RAIZ_PROYECTO / "base" / "localidades_objetivo_sus.gpkg"
RUTA_SALIDA = RAIZ_PROYECTO / "public" / "mapa_base.geojson"

base = gpd.read_file(RUTA_BASE)

In [15]:
base = base[base['tipo_mge'] == 'ageb']

In [20]:
base

,cve_loc,nom_loc,consultorios_faltantes,geometry
109,020010247,El Sauzal de Rodríguez,0.5,"POLYGON ((-116.6887 31.9038, -116.6855 31.9000..."
110,020012183,Rancho Cañón Buena Vista (El Zorrillo),0.2,"POLYGON ((-116.50888 31.67534, -116.51022 31.6..."
113,020020225,Michoacán de Ocampo,0.2,"POLYGON ((-115.3127 32.47564, -115.29986 32.46..."
114,020030001,Tecate,0.2,"POLYGON ((-116.60808 32.57221, -116.60751 32.5..."
123,020041041,Margarita Residencial [Fraccionamiento],0.1,"POLYGON ((-116.77947 32.51816, -116.77855 32.5..."
...,...,...,...,...
2584,150110005,Zapotlán,0.6,"POLYGON ((-98.9059 19.54777, -98.90238 19.5513..."
2585,180060001,Ixtlán del Río,0.0,"POLYGON ((-104.36505 21.04305, -104.36535 21.0..."
2586,203640001,Santa Catarina Juquila,0.3,"POLYGON ((-97.2913 16.24337, -97.28683 16.2405..."
2592,250090001,Escuinapa de Hidalgo,0.1,"POLYGON ((-105.7839 22.83835, -105.78143 22.82..."


In [19]:


columnas_requeridas = {"cve_loc", "nom_loc", "consultorios_faltantes", "geometry"}
columnas_faltantes = columnas_requeridas.difference(base.columns)
if columnas_faltantes:
    raise ValueError(
        f"Faltan columnas requeridas en la base: {sorted(columnas_faltantes)}"
    )

base = base[["cve_loc", "nom_loc", "consultorios_faltantes", "geometry"]].copy()
base["cve_loc"] = base["cve_loc"].astype("string").str.strip()
base["nom_loc"] = base["nom_loc"].astype("string").str.strip()
base["consultorios_faltantes"] = base["consultorios_faltantes"].astype("Float64")
base = base.dropna(subset=["cve_loc", "nom_loc", "geometry"])
base = base[base.geometry.is_valid & ~base.geometry.is_empty]
base = base.drop_duplicates(subset=["cve_loc"], keep="first")

if base.crs is None:
    raise ValueError("La base no tiene CRS definido; no es seguro transformar geometry.")

base = base.to_crs("EPSG:6372")
base["geometry"] = base.geometry.simplify(100, preserve_topology=True)
base = base.to_crs("EPSG:4326")
RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
base.to_file(RUTA_SALIDA, driver="GeoJSON")

print(f"Registros exportados: {len(base):,}")
print(f"CRS de salida: {base.crs}")
print(f"Archivo generado: {RUTA_SALIDA}")

Registros exportados: 1,128
CRS de salida: EPSG:4326
Archivo generado: c:\Users\jose.valdez\Downloads\nuevo-map\mapa-AGEB-\public\mapa_base.geojson
